# Matplotlib

More basic, but generally easier to handle, can be more extensible

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Load the Excel file and the specific sheet
excel_file = r'C:\Users\dane.parks\Downloads\Sample Influence Line_Talmadge Memorial Bridge 1.xlsx'
sheet_name = 'Cable-Axial'

# Read the entire sheet to get the title and data
data = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl', header=None)

# Extract the plot title from A1 (first row, first column explicitly)
plot_title = data.iloc[0, 0]

# Read the sheet again, skipping the first row and using the second row as headers
df = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl', header=1)

# Ensure columns match as described: A -> Plane, B -> X Coordinate, C -> Y Coordinate, D -> Magnitude
df.columns = ['Plane', 'X', 'Y', 'Magnitude']

# Convert the magnitude column to numeric, coercing any non-numeric values to NaN
df['Magnitude'] = pd.to_numeric(df['Magnitude'], errors='coerce')

# Drop rows with missing or NaN values in the Magnitude column
df.dropna(subset=['Magnitude'], inplace=True)

# Get unique planes
planes = df['Plane'].unique()

# Create a single figure for the combined 3D scatter plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Define colors for each plane for better visualization
colors = ['red', 'blue', 'green', 'orange']

# Buffer value for separating lanes
lane_buffer = 10

# Offset tracker for non-overlapping lanes
y_offset = 0

# Iterate through each plane and add to the same 3D plot
for i, plane in enumerate(planes):
    # Isolate data for the current plane
    plane_data = df[df['Plane'] == plane].copy()  # Use copy to avoid modifying original DataFrame

    # Adjust the Y values with the current offset
    plane_data['Y'] += y_offset

    # Add the plane's data to the same 3D plot
    scatter = ax.scatter(
        plane_data['X'],
        plane_data['Y'],
        plane_data['Magnitude'],
        c=colors[i % len(colors)],  # Cycle through the color list
        label=f'Plane {plane}',  # Label the plane for the legend
        s=30  # Size of the points
    )

    # Update the Y offset for the next lane (max Y of current lane + buffer)
    y_offset = plane_data['Y'].max() + lane_buffer

# Set aspect ratio for the X, Y, and Z axes
ax.set_box_aspect([20, 1, 1])  # Custom aspect ratio: 20:1 for x-axis

# Add labels and legend
ax.set_title(plot_title)  # Use the contents of cell A1 as the plot title
ax.set_xlabel('X Coordinate')
ax.set_ylabel('Y Coordinate')
ax.set_zlabel('Magnitude')
ax.legend()  # Show the legend

# Show the plot
plt.show()

In [ ]:
# Load the Excel file and the specific sheet
excel_file = r'C:\Users\dane.parks\Downloads\Sample Influence Line_Talmadge Memorial Bridge 1.xlsx'
sheet_name = 'Cable-Axial'

# Read the entire sheet to get the title and data
data = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl', header=None)

# Extract the plot title from A1 (first row, first column explicitly)
plot_title = data.iloc[0, 0]

# Read the sheet again, skipping the first row and using the second row as headers
df = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl', header=1)

# Ensure columns match as described: A -> Plane, B -> X Coordinate, C -> Y Coordinate, D -> Magnitude
df.columns = ['Plane', 'X', 'Y', 'Magnitude']

# Convert the magnitude column to numeric, coercing any non-numeric values to NaN
df['Magnitude'] = pd.to_numeric(df['Magnitude'], errors='coerce')

# Drop rows with missing or NaN values in the Magnitude column
df.dropna(subset=['Magnitude'], inplace=True)

# Find the row with the maximum magnitude
max_magnitude_row = df.loc[df['Magnitude'].idxmax()]
min_magnitude_row = df.loc[df['Magnitude'].idxmin()]

# Get unique planes
planes = df['Plane'].unique()

# Create a single figure for the combined 3D quiver plot
fig = plt.figure(figsize=(15, 8))  # Larger figure to better display stretched x-axis
ax = fig.add_subplot(111, projection='3d')

# Buffer value for separating lanes
lane_buffer = 10

# Offset tracker for non-overlapping lanes
y_offset = 0

# Iterate through each plane and add arrows to the quiver plot
for i, plane in enumerate(planes):
    # Isolate data for the current plane
    plane_data = df[df['Plane'] == plane].copy()

    # Adjust the Y values with the current offset
    plane_data['Y'] += y_offset

    # Add the quiver arrows for the current plane
    ax.quiver(
        plane_data['X'],  # X-coordinates of the arrow base
        plane_data['Y'],  # Y-coordinates of the arrow base (with offset)
        0,  # Z-coordinates of the arrow base (starts at 0)
        0,  # No X-direction vector component
        0,  # No Y-direction vector component
        plane_data['Magnitude'],  # Z-direction vector (force magnitude as arrow length)
        arrow_length_ratio=0.15,  # Make arrowheads clearer and proportional
        color='blue',  # Blue arrows for visual clarity
        linewidths=0.5  # Thin arrow lines
    )

    # Update the Y offset for the next plane (leave space between planes)
    y_offset = plane_data['Y'].max() + lane_buffer

# Set stretched aspect ratio for consistency with 20:1 X-axis scaling
ax.set_box_aspect([20, 1, 1])  # Stretch X axis to 20:1 relative to Y and Z

# Label the maximum Z value (Magnitude) with its X, Y, Z coordinates and lane
ax.text(
    max_magnitude_row['X'],  # X-coordinate of the maximum magnitude
    max_magnitude_row['Y'],  # Y-coordinate of the maximum magnitude
    max_magnitude_row['Magnitude'],  # Z-coordinate (maximum magnitude value)
    f"Max: {max_magnitude_row['Magnitude']:.2f}\n"
    f"Lane: {max_magnitude_row['Plane']},\n"  # Include the lane (plane) in the label
    f"X: {max_magnitude_row['X']:.2f}, Y: {max_magnitude_row['Y']:.2f}, Z: {max_magnitude_row['Magnitude']:.2f}",
    # Coordinates and magnitude
    color='red',  # Color for the label
    fontsize=10,
    weight='bold'
)

ax.text(
    min_magnitude_row['X'],  # X-coordinate of the maximum magnitude
    min_magnitude_row['Y'],  # Y-coordinate of the maximum magnitude
    min_magnitude_row['Magnitude'],  # Z-coordinate (maximum magnitude value)
    f"Max: {min_magnitude_row['Magnitude']:.2f}\n"
    f"Lane: {min_magnitude_row['Plane']},\n"  # Include the lane (plane) in the label
    f"X: {min_magnitude_row['X']:.2f}, Y: {min_magnitude_row['Y']:.2f}, Z: {min_magnitude_row['Magnitude']:.2f}",
    # Coordinates and magnitude
    color='blue',  # Color for the label
    fontsize=10,
    weight='bold'
)

# Set the plot labels and title
ax.set_title(plot_title, fontsize=14)  # Use the contents of cell A1 as the plot title
ax.set_xlabel('X Coordinate', fontsize=12)
ax.set_ylabel('Y Coordinate', fontsize=12)
ax.set_zlabel('Magnitude', fontsize=12)

# Adjust the view angle for better visualization (optional)
ax.view_init(elev=20, azim=45)

# Display the 3D plot
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import use

# Set matplotlib to use the widget backend for interactivity
use('webagg')  # For a browser-based interactive plot (or use 'qt5agg' if running locally)

# Load the Excel file and the specific sheet
excel_file = r'C:\Users\dane.parks\Downloads\Sample Influence Line_Talmadge Memorial Bridge 1.xlsx'
sheet_name = 'Cable-Axial'

# Read the entire sheet to get the title and data
data = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl', header=None)

# Extract the plot title from A1 (first row, first column explicitly)
plot_title = data.iloc[0, 0]

# Read the sheet again, skipping the first row and using the second row as headers
df = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl', header=1)

# Ensure columns match as described: A -> Plane, B -> X Coordinate, C -> Y Coordinate, D -> Magnitude
df.columns = ['Plane', 'X', 'Y', 'Magnitude']

# Convert the magnitude column to numeric, coercing any non-numeric values to NaN
df['Magnitude'] = pd.to_numeric(df['Magnitude'], errors='coerce')

# Drop rows with missing or NaN values in the Magnitude column
df.dropna(subset=['Magnitude'], inplace=True)

# Scale magnitude by a factor of 5
df['Magnitude'] /= 5  # Scaling down the magnitudes for visualization

# Get unique planes
planes = df['Plane'].unique()

# Create a single figure for the combined 3D quiver plot
fig = plt.figure(figsize=(15, 8))  # Larger figure for more clarity
ax = fig.add_subplot(111, projection='3d')

# Buffer value for separating lanes
lane_buffer = 10

# Offset tracker for non-overlapping lanes
y_offset = 0

# Iterate through each plane and add arrows to the quiver plot
for i, plane in enumerate(planes):
    # Isolate data for the current plane
    plane_data = df[df['Plane'] == plane].copy()

    # Adjust the Y values with the current offset
    plane_data['Y'] += y_offset

    # Add the quiver arrows for the current plane
    ax.quiver(
        plane_data['X'],  # X-coordinates of the arrow base
        plane_data['Y'],  # Y-coordinates of the arrow base (with offset)
        0,  # Z-coordinates of the arrow base (starts at 0)
        0,  # No X-direction vector component
        0,  # No Y-direction vector component
        plane_data['Magnitude'],  # Z-direction vector (force magnitude as arrow length)
        arrow_length_ratio=0.15,  # Adjust size of arrowheads
        color='blue',  # Blue arrows
        linewidths=0.5  # Thin arrow lines
    )

    # Update the Y offset for the next lane (leave space between planes)
    y_offset = plane_data['Y'].max() + lane_buffer

# Set stretched aspect ratio for consistency with 20:1 X-axis scaling
ax.set_box_aspect([20, 1, 1])  # Stretch X axis to 20:1 relative to Y and Z

# Set the plot labels and title
ax.set_title(plot_title, fontsize=14)  # Use the contents of cell A1 as the plot title
ax.set_xlabel('X Coordinate', fontsize=12)
ax.set_ylabel('Y Coordinate', fontsize=12)
ax.set_zlabel('Magnitude (Scaled by 1/5)', fontsize=12)

# Interactive zoom and pan work with backend; elev and azim can still be set
ax.view_init(elev=20, azim=45)

# Display the interactive plot
plt.show()

# Plotly

Better looking 

In [ ]:
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots

# Load the Excel file and the specific sheet
excel_file = r'C:\Users\dane.parks\Downloads\Sample Influence Line_Talmadge Memorial Bridge 1.xlsx'
sheet_name = 'Cable-Axial'

# Read the entire sheet to get the title and data
data = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl', header=None)

# Extract the plot title from A1 (first row, first column)
plot_title = data.iloc[0, 0]

# Read the sheet again, skipping the first row and using the second row as headers
df = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl', header=1)

# Ensure columns match as described: A -> Plane, B -> X Coordinate, C -> Y Coordinate, D -> Magnitude
df.columns = ['Plane', 'X', 'Y', 'Magnitude']

# Convert the magnitude column to numeric, coercing any non-numeric values to NaN
df['Magnitude'] = pd.to_numeric(df['Magnitude'], errors='coerce')

# Drop rows with missing or NaN values in the Magnitude column
df.dropna(subset=['Magnitude'], inplace=True)

# Get unique planes
planes = df['Plane'].unique()

# Define a buffer value to separate lanes
lane_buffer = 10

# Offset tracker for non-overlapping lanes
y_offset = 0

# Create a list for 3D scatter traces
traces = []

# Iterate through each plane and create a trace for Plotly
for i, plane in enumerate(planes):
    # Isolate data for the current plane
    plane_data = df[df['Plane'] == plane].copy()  # Use copy to avoid modifying original DataFrame

    # Adjust the Y values with the current offset
    plane_data['Y'] += y_offset

    # Create 3D scatter trace
    trace = go.Scatter3d(
        x=plane_data['X'],
        y=plane_data['Y'],
        z=plane_data['Magnitude'],
        mode='markers',
        marker=dict(
            size=5,
            color=i,  # Use the index for coloring (unique for each plane)
            colorscale='Viridis',  # Use a colorscale for better visualization
            opacity=0.8
        ),
        name=f'Plane {plane}'  # Add a name/label for the legend
    )
    traces.append(trace)

    # Update the Y offset for the next lane (max Y of current lane + buffer)
    y_offset = plane_data['Y'].max() + lane_buffer

# Create the layout
layout = go.Layout(
    title=dict(
        text=plot_title,  # Dynamically set the title from Cell A1
        x=0.5  # Center the title
    ),
    scene=dict(
        xaxis_title='X Coordinate',
        yaxis_title='Y Coordinate',
        zaxis_title='Magnitude',
        aspectratio=dict(
            x=20,  # X-axis is 20 times longer
            y=1,  # Y-axis
            z=1  # Z-axis
        )
    ),
    legend=dict(
        itemsizing='constant',
        x=0.8,
        y=1
    )
)

# Combine the traces and layout into a figure
fig = go.Figure(data=traces, layout=layout)

# Display the plot
fig.show()

In [ ]:
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots

# Load the Excel file and the specific sheet
excel_file = r'C:\Users\dane.parks\Downloads\Sample Influence Line_Talmadge Memorial Bridge 1.xlsx'
sheet_name = 'Cable-Axial'

# Read the entire sheet to get the title and data
data = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl', header=None)

# Extract the plot title from A1 (first row, first column)
plot_title = data.iloc[0, 0]

# Read the sheet again, skipping the first row and using the second row as headers
df = pd.read_excel(excel_file, sheet_name=sheet_name, engine='openpyxl', header=1)

# Ensure columns match as described: A -> Plane, B -> X Coordinate, C -> Y Coordinate, D -> Magnitude
df.columns = ['Plane', 'X', 'Y', 'Magnitude']

# Convert the magnitude column to numeric, coercing any non-numeric values to NaN
df['Magnitude'] = pd.to_numeric(df['Magnitude'], errors='coerce')

# Drop rows with missing or NaN values in the Magnitude column
df.dropna(subset=['Magnitude'], inplace=True)

# Find the rows with the maximum and minimum magnitudes
max_magnitude_row = df.loc[df['Magnitude'].idxmax()]
min_magnitude_row = df.loc[df['Magnitude'].idxmin()]

# Get unique planes
planes = df['Plane'].unique()

# Define a buffer value to separate lanes
lane_buffer = 10

# Offset tracker for non-overlapping lanes
y_offset = 0

# Create a list for 3D scatter traces
traces = []

# Create a dictionary to map planes to their respective offsets
plane_offsets = {}

# Iterate through each plane and create a trace for Plotly
for i, plane in enumerate(planes):
    # Isolate data for the current plane
    plane_data = df[df['Plane'] == plane].copy()  # Use copy to avoid modifying original DataFrame

    # Adjust the Y values with the current offset
    plane_data['Y'] += y_offset

    # Store the offset for the current plane
    plane_offsets[plane] = y_offset

    # Create 3D scatter trace
    trace = go.Scatter3d(
        x=plane_data['X'],
        y=plane_data['Y'],
        z=plane_data['Magnitude'],
        mode='markers',
        marker=dict(
            size=5,
            color=i,  # Use the index for coloring (unique for each plane)
            colorscale='Viridis',  # Use a colorscale for better visualization
            opacity=0.8
        ),
        name=f'Plane {plane}'  # Add a name/label for the legend
    )
    traces.append(trace)

    # Update the Y offset for the next lane (max Y of current lane + buffer)
    y_offset = plane_data['Y'].max() + lane_buffer

# Correct the Y value for the maximum annotation by adding the lane-specific y_offset
adjusted_y_max = max_magnitude_row['Y'] + plane_offsets[max_magnitude_row['Plane']]

# Correct the Y value for the minimum annotation by adding the lane-specific y_offset
adjusted_y_min = min_magnitude_row['Y'] + plane_offsets[min_magnitude_row['Plane']]

# Create an annotation for the maximum value
max_annotation = {
    'x': max_magnitude_row['X'],  # X-coordinate of the maximum point
    'y': adjusted_y_max,  # Adjusted Y-coordinate for the maximum point
    'z': max_magnitude_row['Magnitude'],  # Magnitude (Z) of the maximum point
    'text': f"Max: {max_magnitude_row['Magnitude']:.2f}<br>"
            f"Plane: {max_magnitude_row['Plane']}<br>"
            f"X: {max_magnitude_row['X']:.2f}, Y: {adjusted_y_max:.2f}",
    'showarrow': True,
    'arrowhead': 2,
    'arrowcolor': 'red',
    # Adjust offsets so the annotation is near the maximum but not misplaced
    'ax': 0,  # Keeps the arrow horizontally aligned
    'ay': -100,  # Move the annotation text above the lane (away from the arrow target)
    'font': dict(
        color='red',
        size=12
    )
}

# Create an annotation for the minimum value
min_annotation = {
    'x': min_magnitude_row['X'],  # X-coordinate of the minimum point
    'y': adjusted_y_min,  # Adjusted Y-coordinate for the minimum point
    'z': min_magnitude_row['Magnitude'],  # Magnitude (Z) of the minimum point
    'text': f"Min: {min_magnitude_row['Magnitude']:.2f}<br>"
            f"Plane: {min_magnitude_row['Plane']}<br>"
            f"X: {min_magnitude_row['X']:.2f}, Y: {adjusted_y_min:.2f}",
    'showarrow': True,
    'arrowhead': 2,
    'arrowcolor': 'blue',
    # Adjust offsets so the annotation is near the minimum but not misplaced
    'ax': 0,  # Keeps the arrow horizontally aligned
    'ay': 100,  # Move the annotation text below the lane (away from the arrow target)
    'font': dict(
        color='blue',
        size=12
    )
}

# Create the layout
layout = go.Layout(
    title=dict(
        text=plot_title,  # Dynamically set the title from Cell A1
        x=0.5  # Center the title
    ),
    scene=dict(
        xaxis_title='X Coordinate',
        yaxis_title='Y Coordinate',
        zaxis_title='Magnitude',
        annotations=[max_annotation, min_annotation],  # Add both annotations to the layout
        aspectratio=dict(
            x=20,  # X-axis is 20 times longer
            y=1,  # Y-axis
            z=1  # Z-axis
        )
    ),
    legend=dict(
        itemsizing='constant',
        x=0.8,
        y=1
    )
)

# Combine the traces and layout into a figure
fig = go.Figure(data=traces, layout=layout)

# Display the plot
fig.show()

# Extending the Data

All bridge data reported to FHWA is available [here.](https://www.fhwa.dot.gov/bridge/nbi/ascii.cfm)

That means, it's available to python,

In [ ]:
from civilpy.state.ohio.DOT.legacy import get_historic_bridge_data

In [ ]:
sfn='000000005101690'

In [ ]:
import io
import requests

def get_df_from_url(url):
    r = requests.get(url)
    if r.ok:
        df = pd.read_csv(io.BytesIO(r.content), low_memory=False, quotechar="'")
    else:
        # print(f"Couldn't find a dataframe at {url} make sure this page is still up")
        df = None

    return df

In [ ]:
nbi_df = get_df_from_url(
            "https://daneparks.com/Dane/civilpy/-/raw/Data/res/2022AllRecordsDelimitedAllStates.txt"
        )

In [ ]:
nbi_df

In [ ]:
first_bridge_data = nbi_df[nbi_df["STRUCTURE_NUMBER_008"].str.strip() == sfn]

In [ ]:
talmadge_data = first_bridge_data.head(1)

In [ ]:
lat, lon = talmadge_data['LAT_016'].values[0]/1000000, talmadge_data['LONG_017'].values[0]/1000000

In [ ]:
import folium

In [ ]:
bridge_map = folium.Map( location=[lat, -lon], zoom_start=6 )
folium.Marker( location=[ lat, -lon ], fill_color='#43d9de', radius=8 ).add_to( bridge_map )
bridge_map

In [ ]:
import folium
from branca.element import Html
import pandas as pd
import plotly.graph_objects as go

# Your existing Plotly figure generation code remains unchanged
# Generate your Plotly graph:
# -------------------------------------------------
# Copy and paste your Plotly graph code, up to where the `fig` is generated
# -------------------------------------------------

# Export the Plotly graph to HTML (as a string)
plotly_html = fig.to_html(full_html=False, include_plotlyjs='cdn')

# Create a Folium map
lat, lon = 32.0811, -81.0912  # Replace with actual lat/lon values
bridge_map = folium.Map(location=[lat, lon], zoom_start=6)

# Create a custom HTML snippet that embeds the Plotly HTML
html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
</head>
<body>
    {plotly_html}  <!-- Embed the Plotly HTML directly here -->
</body>
</html>
"""

# Wrap the HTML in a Branca element and make it into a popup iframe
iframe = folium.IFrame(html=html, width=650, height=500)
popup = folium.Popup(iframe, max_width=1000)

# Add a marker to the Folium map
marker = folium.Marker(
    location=[lat, lon],
    popup=popup,
    tooltip="Talmadge Bridge"
)
marker.add_to(bridge_map)

tile = folium.TileLayer(
        tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr = 'Esri',
        name = 'Esri Satellite',
        overlay = False,
        control = True
       ).add_to(bridge_map)

# Display or save the map to an HTML file
bridge_map.save("bridge_map_with_plotly.html")  # Save and open this in your browser
bridge_map

# Other Data Available

In [ ]:
print(f"Deck Structure Type: {talmadge_data['DECK_STRUCTURE_TYPE_107'].values[0]}")

In [ ]:
print(f"Deck Surface Type: {talmadge_data['SURFACE_TYPE_108A'].values[0]}")

In [ ]:
print(f"Structure Type: {talmadge_data['RECORD_TYPE_005A'].values[0]}")

In [ ]:
print(f"Owner: {talmadge_data['OWNER_022'].values[0]}")

In [ ]:
print(f"Route: {talmadge_data['ROUTE_NUMBER_005D'].values[0]}")

In [ ]:
print(f"Max span length: {talmadge_data['MAX_SPAN_LEN_MT_048'].values[0]}")

In [ ]:
print(f"Structure length: {talmadge_data['STRUCTURE_LEN_MT_049'].values[0]}")

In [ ]:
print(f"Deck width: {talmadge_data['DECK_WIDTH_MT_052'].values[0]}")

In [ ]:
print(f"Spans: {talmadge_data['MAIN_UNIT_SPANS_045'].values[0]}")

In [ ]:
print(f"Operating Rating: {talmadge_data['OPERATING_RATING_064'].values[0]}")

In [ ]:
print(f"Inventory Rating: {talmadge_data['INVENTORY_RATING_066'].values[0]}")

In [ ]:
print(f"Submitted by: {talmadge_data['SUBMITTED_BY'].values[0]}")

In [ ]:
print(f"Fed Agency: {talmadge_data['FED_AGENCY'].values[0]}")

In [ ]:
all_columns = talmadge_data.columns.to_list()